## 지난주 개념 복습!
아래 마크다운을 풀고, 지난주 예습과제를 통해 공부한 Vision Transformer의 특징과 다른 모델들과의 차이점을 간략히 설명해주세요.

- ViT는 이미지를 일정한 크기의 패치로 나눈 뒤 각 패치를 토큰처럼 처리하고, 패치 임베딩과 위치 임베딩을 Transformer Encoder에 입력해 Self-Attention으로 패치 사이의 관계를 학습합니다. CNN처럼 합성곱을 이용해 지역적인 특징을 우선 추출하는 구조가 아니기 때문에 이미지에 대한 귀납적 편향이 상대적으로 적고, 분류에는 [CLS] 토큰을 사용합니다.

- Swin Transformer는 전체 패치에 한 번에 Attention을 적용하는 ViT와 달리 Window 단위의 Attention과 Shifted Window를 사용하고, Patch Merging을 통해 계층적인 특징을 학습합니다. CvT는 합성곱 토큰 임베딩과 합성곱 기반 Q/K/V Projection을 사용해 CNN의 지역적 특징 추출과 Transformer의 관계 학습을 결합하며, 별도의 위치 임베딩을 사용하지 않습니다.

## 파이토치로 구현한 Vision Transformer
이번 복습과제에서는 카사바 잎 질병 데이터셋으로 vision transformer를 학습시키겠습니다.
- 첨부된 드라이브에서 필요한 파일들을 다운받을 수 있습니다.

In [1]:
!curl https://raw.githubusercontent.com/pytorch/xla/master/contrib/scripts/env-setup.py -o pytorch-xla-env-setup.py
!python pytorch-xla-env-setup.py --version 1.7
!pip install timm
!pip install torch_xla

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    14  100    14    0     0     84      0 --:--:-- --:--:-- --:--:--    84
  File "/content/pytorch-xla-env-setup.py", line 1
    404: Not Found
    ^^^
SyntaxError: illegal target for annotation
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 MB 47.3 MB/s  0:00:01


### 데이터 불러오기

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#.zip 파일 unzip
#본인 드라이브의 경로에 맞게 수정해주세요!
!unzip /content/drive/MyDrive/mydata/train_images -d /content/drive/MyDrive/mydata/train_images

Archive:  /content/drive/MyDrive/mydata/train_images.zip
replace /content/drive/MyDrive/mydata/train_images/1000015157.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: N
N
N
N

N
  inflating: /content/drive/MyDrive/mydata/train_images/3527165610.jpg  


In [4]:
#본인 드라이브의 경로에 맞게 수정해주세요!
!unzip /content/drive/MyDrive/mydata/jx_vit_base_p16_224-80ecf9dd.pth.zip -d /content/drive/MyDrive/mydata/

Archive:  /content/drive/MyDrive/mydata/jx_vit_base_p16_224-80ecf9dd.pth.zip
replace /content/drive/MyDrive/mydata/jx_vit_base_p16_224-80ecf9dd.pth? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [5]:
#파일 경로
DATA_PATH = "/content/drive/MyDrive/mydata/"
TRAIN_PATH = "/content/drive/MyDrive/mydata/train_images"
TEST_PATH = "/content/drive/MyDrive/mydata/test.jpg"
MODEL_PATH = (
    "/content/drive/MyDrive/mydata/jx_vit_base_p16_224-80ecf9dd.pth"
)

### 라이브러리

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("ggplot")

import torch
import torch.nn as nn
import torchvision.transforms as transforms

import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.distributed.xla_multiprocessing as xmp
import torch_xla.distributed.parallel_loader as pl

import timm

import gc
import os
import time
import random
from datetime import datetime

from PIL import Image
from tqdm.notebook import tqdm
from sklearn import model_selection, metrics

ImportError: /usr/local/lib/python3.13/dist-packages/_XLAC.cpython-313-x86_64-linux-gnu.so: undefined symbol: _ZN5torch8autograd10deleteNodeEPNS0_4NodeE

In [ ]:
os.environ["XLA_USE_BF16"] = "1"
os.environ["XLA_TENSOR_ALLOCATOR_MAXSIZE"] = "100000000"

In [ ]:
def seed_everything(seed):
    """
    Seeds basic parameters for reproductibility of results

    Arguments:
        seed {int} -- Number of the seed
    """
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(1001)

### 파라미터

In [ ]:
IMG_SIZE = 224  #ViT에 적합한 이미지 크기로 설정해주세요.
BATCH_SIZE = 16
LR = 2e-05
GAMMA = 0.7
N_EPOCHS = 1  #모델 학습을 위해서는 더 많은 에포크가 필요하지만, 파일 크기가 큰 관계로 이번 과제에서는 전 과정을 한 번만 수행하겠습니다.

In [ ]:
# pandas로 csv 파일을 읽은 뒤, 첫 5행을 출력하세요.
df =  pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
df.head()

In [ ]:
df.info()

In [ ]:
df.label.value_counts().plot(kind="bar")

In [1]:
# train과 test를 9:1로 나눠주세요.
# random_state은 자유롭게 지정 가능합니다.
train_df, valid_df = model_selection.train_test_split(
    df, test_size=0.1, random_state=42, stratify=df.label.values
)

NameError: name 'model_selection' is not defined

In [ ]:
class CassavaDataset(torch.utils.data.Dataset):

    def __init__(self, df, data_path=DATA_PATH, mode="train", transforms=None):
        super().__init__()
        self.df_data = df.values
        self.data_path = data_path
        self.transforms = transforms
        self.mode = mode
        self.data_dir = "train_images" if mode == "train" else "test_images"

    def __len__(self):
        return len(self.df_data)

    def __getitem__(self, index):
        img_name, label = self.df_data[index]
        img_path = os.path.join(self.data_path, self.data_dir, img_name)
        img = Image.open(img_path).convert("RGB")

        if self.transforms is not None:
            image = self.transforms(img)

        return image, label

In [ ]:
#데이터 증강
transforms_train = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.3),
        transforms.RandomResizedCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
    ]
)

transforms_valid = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
    ]
)

In [ ]:
#사용 가능한 ViT 모델 목록
##우리는 이중 vit_base_patch16_224를 사용합니다!
print("Available Vision Transformer Models: ")
timm.list_models("vit*")

### 모델 설계
- 클래스 ViTBase16의 각 줄을 설명하는 주석을 달아주세요. (최소 15개)

In [ ]:
# 이 셀 안의 코드에 대해서 주석을 달아주세요.

# ViT-Base/16을 이용한 이미지 분류 모델을 정의
class ViTBase16(nn.Module):

    # 클래스 수와 사전 학습 가중치 사용 여부를 설정
    def __init__(self, n_classes, pretrained=False):

        super(ViTBase16, self).__init__()

        # timm에서 ViT-Base/16 모델을 생성
        self.model = timm.create_model("vit_base_patch16_224", pretrained=False)

        # pretrained=True인 경우 저장된 사전 학습 가중치를 불러옴
        if pretrained:
            self.model.load_state_dict(torch.load(MODEL_PATH))

        # 출력층을 현재 데이터의 클래스 수에 맞게 변경
        self.model.head = nn.Linear(self.model.head.in_features, n_classes)

    # 입력 이미지를 ViT 모델에 통과시켜 예측값을 반환
    def forward(self, x):
        x = self.model(x)
        return x

    # 한 epoch 동안 학습을 수행하고 평균 loss와 accuracy를 계산
    def train_one_epoch(self, train_loader, criterion, optimizer, device):
        epoch_loss = 0.0
        epoch_accuracy = 0.0

        # 모델을 학습 모드로 설정하고 미니배치 단위로 학습
        self.model.train()
        for i, (data, target) in enumerate(train_loader):

            # 사용 환경에 따라 데이터를 GPU 또는 TPU로 이동
            if device.type == "cuda":
                data, target = data.cuda(), target.cuda()
            elif device.type == "xla":
                data = data.to(device, dtype=torch.float32)
                target = target.to(device, dtype=torch.int64)

            # gradient 초기화 → 순전파 → loss 계산 → 역전파 수행
            optimizer.zero_grad()
            output = self.forward(data)
            loss = criterion(output, target)
            loss.backward()

            # 예측 결과를 이용해 accuracy를 계산
            accuracy = (output.argmax(dim=1) == target).float().mean()

            # 각 배치의 loss와 accuracy를 누적
            epoch_loss += loss
            epoch_accuracy += accuracy

            # TPU에서는 XLA optimizer를 사용해 파라미터를 업데이트
            if device.type == "xla":
                xm.optimizer_step(optimizer)

                # 20개 배치마다 현재 loss를 출력
                if i % 20 == 0:
                    xm.master_print(f"\tBATCH {i+1}/{len(train_loader)} - LOSS: {loss}")

            # TPU가 아닌 경우 일반 optimizer를 사용
            else:
                optimizer.step()

        # 전체 학습 배치의 평균 loss와 accuracy를 반환
        return epoch_loss / len(train_loader), epoch_accuracy / len(train_loader)

    # validation 데이터로 모델의 loss와 accuracy를 평가
    def validate_one_epoch(self, valid_loader, criterion, device):
        valid_loss = 0.0
        valid_accuracy = 0.0

        # 모델을 평가 모드로 설정
        self.model.eval()
        for data, target in valid_loader:

            # 사용 환경에 따라 validation 데이터를 GPU 또는 TPU로 이동
            if device.type == "cuda":
                data, target = data.cuda(), target.cuda()
            elif device.type == "xla":
                data = data.to(device, dtype=torch.float32)
                target = target.to(device, dtype=torch.int64)

            # 검증에서는 gradient 계산 없이 예측, loss, accuracy만 계산
            with torch.no_grad():
                output = self.model(data)
                loss = criterion(output, target)
                accuracy = (output.argmax(dim=1) == target).float().mean()

                valid_loss += loss
                valid_accuracy += accuracy

        # 전체 validation 배치의 평균 loss와 accuracy를 반환
        return valid_loss / len(valid_loader), valid_accuracy / len(valid_loader)

In [ ]:
def fit_tpu(
    model, epochs, device, criterion, optimizer, train_loader, valid_loader=None
):

    valid_loss_min = np.inf

    train_losses = []
    valid_losses = []
    train_accs = []
    valid_accs = []

    for epoch in range(1, epochs + 1):
        gc.collect()
        para_train_loader = pl.ParallelLoader(train_loader, [device])

        xm.master_print(f"{'='*50}")
        xm.master_print(f"EPOCH {epoch} - TRAINING...")
        train_loss, train_acc = model.train_one_epoch(
            para_train_loader.per_device_loader(device), criterion, optimizer, device
        )
        xm.master_print(
            f"\n\t[TRAIN] EPOCH {epoch} - LOSS: {train_loss}, ACCURACY: {train_acc}\n"
        )
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        gc.collect()

        if valid_loader is not None:
            gc.collect()
            para_valid_loader = pl.ParallelLoader(valid_loader, [device])
            xm.master_print(f"EPOCH {epoch} - VALIDATING...")
            valid_loss, valid_acc = model.validate_one_epoch(
                para_valid_loader.per_device_loader(device), criterion, device
            )
            xm.master_print(f"\t[VALID] LOSS: {valid_loss}, ACCURACY: {valid_acc}\n")
            valid_losses.append(valid_loss)
            valid_accs.append(valid_acc)
            gc.collect()

            if valid_loss <= valid_loss_min and epoch != 1:
                xm.master_print(
                    "Validation loss decreased ({:.4f} --> {:.4f}).  Saving model ...".format(
                        valid_loss_min, valid_loss
                    )
                )

            valid_loss_min = valid_loss

    return {
        "train_loss": train_losses,
        "valid_losses": valid_losses,
        "train_acc": train_accs,
        "valid_acc": valid_accs,
    }

In [ ]:
model = ViTBase16(n_classes=5, pretrained=True)

In [ ]:
def _run():
    train_dataset = CassavaDataset(train_df, transforms=transforms_train)
    valid_dataset = CassavaDataset(valid_df, transforms=transforms_valid)

    train_sampler = torch.utils.data.distributed.DistributedSampler(
        train_dataset,
        num_replicas=xm.xrt_world_size(),
        rank=xm.get_ordinal(),
        shuffle=True,
    )

    valid_sampler = torch.utils.data.distributed.DistributedSampler(
        valid_dataset,
        num_replicas=xm.xrt_world_size(),
        rank=xm.get_ordinal(),
        shuffle=False,
    )

    train_loader = torch.utils.data.DataLoader(
        dataset=train_dataset,
        batch_size=BATCH_SIZE,
        sampler=train_sampler,
        drop_last=True,
        num_workers=8,
    )

    valid_loader = torch.utils.data.DataLoader(
        dataset=valid_dataset,
        batch_size=BATCH_SIZE,
        sampler=valid_sampler,
        drop_last=True,
        num_workers=8,
    )

    criterion =  nn.CrossEntropyLoss()   #cross entropy loss로 설정.
    device = xm.xla_device()
    model.to(device)

    lr = LR * xm.xrt_world_size()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    xm.master_print(f"INITIALIZING TRAINING ON {xm.xrt_world_size()} TPU CORES")
    start_time = datetime.now()
    xm.master_print(f"Start Time: {start_time}")

    logs = fit_tpu(
        model=model,
        epochs=N_EPOCHS,
        device=device,
        criterion=criterion,
        optimizer=optimizer,
        train_loader=train_loader,
        valid_loader=valid_loader,
    )

    xm.master_print(f"Execution time: {datetime.now() - start_time}")

    xm.master_print("Saving Model")
    xm.save(
        model.state_dict(), f'model_5e_{datetime.now().strftime("%Y%m%d-%H%M")}.pth'
    )

In [ ]:
# Training 시작
def _mp_fn(rank, flags):
    torch.set_default_tensor_type("torch.FloatTensor")
    a = _run()


# Run
FLAGS = {}
xmp.spawn(_mp_fn, args=(FLAGS,), nprocs=1, start_method="fork")